# 01 — Inspeção dos datasets

Rode isto **antes** de escrever qualquer código de treino. O objetivo é confirmar que os
números da sua cópia batem com os auditados no plano, e enxergar as armadilhas com os
próprios olhos.

Se algum número divergir, **pare** e descubra por quê antes de continuar.

In [ ]:
import os, sys, json
sys.path.insert(0, '..')
os.environ.setdefault('TCC_DATA_ROOT', 'D:/Erick/Dataset')
from src.data.vindr import load_findings, sanity_check

df = load_findings()
print(json.dumps(sanity_check(df), indent=2, ensure_ascii=False))
# Esperado: 20486 linhas, 2254 com caixa, training 16391 / test 4095,
#           finding_birads apenas 3/4/5

In [ ]:
# Desbalanceamento: apenas ~4,9% das mamas sao BI-RADS 4-5; densidade A tem 100 linhas.
print(df.breast_birads.value_counts())
print(df.breast_density.value_counts())

In [ ]:
# finding_categories e MULTI-ROTULO e vem como string de lista Python.
from collections import Counter
c = Counter(tuple(x) for x in df.loc[df.has_box, 'categories'])
for k, v in c.most_common(12):
    print(v, k)

In [ ]:
# CBIS-DDSM: vazamento cruzado massa <-> calcificacao nos splits oficiais.
from src.data.cbis import load_all, leakage_report
cb = load_all()
print(len(cb), 'anormalidades |', cb.patient_id.nunique(), 'pacientes')
print(json.dumps(leakage_report(cb), indent=2))

In [ ]:
# INbreast: laudos em PORTUGUES, encoding latin-1.
from src.data.inbreast import load_reports as inb_reports
ir = inb_reports()
print(len(ir), 'laudos |', ir.patient.nunique(), 'pacientes')
print(ir.birads_text.value_counts(dropna=False))
print('\n---\n', ir.iloc[0].text[:400])

In [ ]:
# Extracao por regras em portugues (INbreast -- unico corpus de laudo livre do projeto).
from src.report.extract_rules import RuleExtractor
rec_pt = RuleExtractor('pt').extract(ir.iloc[0].text, ir.iloc[0].file, 'R')
print([(f.category, f.status.value) for f in rec_pt.findings], rec_pt.breast_birads)
print(rec_pt.model_dump_json(indent=2, exclude_none=True)[:1200])
